# CamusGPT v2 — Phase 1: Voice SFT on Gemma-3-12B

Trains the **voice** LoRA on `unsloth/gemma-3-12b-it` from the tracked corpus. Output:
`adapters/camus_v2_voice_lora` on Drive — the INPUT to the Step-3.5 refusal-SFT notebook.

**Why a new Phase 1:** LoRA adapters are base-specific. The Llama-3.1-8B adapter cannot be
reused on Gemma 3; both passes are re-run from the same corpus (all tracked in the repo).

**Gemma-3 differences that matter here (vs the Llama runs):**
1. **Chat template.** Gemma uses `<start_of_turn>user ... <end_of_turn>` / `<start_of_turn>model`
   and has **no system role** — a system prompt must be folded into the first user turn.
   Unsloth's `gemma-3` chat template handles this; we verify it rather than trust it.
2. **Response masking.** The assistant marker is `<start_of_turn>model\n`, NOT Llama's
   `<|start_header_id|>assistant<|end_header_id|>`. CHECK 2 below verifies masking against
   the real token ids — this is THE critical gate; wrong offsets train on the prompt.
3. **Attention/dtype.** Gemma 3 wants bf16 and eager attention for stability on A100/L4.

Runtime: **A100 (40GB) recommended**; L4 works with the smaller batch shown.

Needs on Drive at `MyDrive/CamusGPT_Training/data/`:
`camus_sft.jsonl`, `camus_conversational.jsonl`


In [ ]:
# 1) Environment
!pip install -q --no-deps unsloth vllm==0.* 2>/dev/null
!pip install -q unsloth
import torch, transformers, unsloth
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| unsloth", unsloth.__version__)
print("GPU:", torch.cuda.get_device_name(0),
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

In [ ]:
# 2) Drive + config
from google.colab import drive
drive.mount('/content/drive')

DRIVE      = "/content/drive/MyDrive/CamusGPT_Training"
BASE_MODEL = "unsloth/gemma-3-12b-it"
OUT_ADAPTER= f"{DRIVE}/adapters/camus_v2_voice_lora"
MAX_SEQ    = 2048

import os
for p in [f"{DRIVE}/data/camus_sft.jsonl", f"{DRIVE}/data/camus_conversational.jsonl"]:
    assert os.path.exists(p), f"MISSING: {p}"
    print("ok:", p, os.path.getsize(p), "bytes")
os.makedirs(f"{DRIVE}/adapters", exist_ok=True)

In [ ]:
# 3) Load Gemma-3-12B + attach a fresh LoRA
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,          # QLoRA: 12B fits comfortably on A100/L4
    dtype = None,                 # auto -> bf16 where supported
)
model = FastModel.get_peft_model(
    model,
    r = 32,                       # voice pass: a little more capacity than the guardrail pass
    lora_alpha = 32,
    lora_dropout = 0.0,
    bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print(model.print_trainable_parameters())

In [ ]:
# 4) Chat template — Gemma 3 has NO system role; verify before training
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

probe = tokenizer.apply_chat_template(
    [{"role":"user","content":"PROMPT_HERE"},
     {"role":"assistant","content":"RESPONSE_HERE"}],
    tokenize=False, add_generation_prompt=False)
print(probe)

assert "<start_of_turn>model" in probe, "unexpected template: no model turn marker"
assert "PROMPT_HERE" in probe and "RESPONSE_HERE" in probe
print("\nCHECK 0 PASSED — gemma-3 template applied")

In [ ]:
# 5) Load the corpus -> {"prompt","response"} pairs
import json, random
from collections import Counter

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def to_pair(r):
    """Accept the several shapes used across this project's corpora."""
    if "prompt" in r and "response" in r:
        return {"prompt": r["prompt"], "response": r["response"]}
    if "instruction" in r and "output" in r:
        return {"prompt": r["instruction"], "response": r["output"]}
    if "messages" in r:
        u = next((m["content"] for m in r["messages"] if m["role"] == "user"), None)
        a = next((m["content"] for m in r["messages"] if m["role"] == "assistant"), None)
        if u and a:
            return {"prompt": u, "response": a}
    if "completion" in r and "prompt" in r:
        return {"prompt": r["prompt"], "response": r["completion"]}
    return None

sft  = [p for p in map(to_pair, load_jsonl(f"{DRIVE}/data/camus_sft.jsonl")) if p]
conv = [p for p in map(to_pair, load_jsonl(f"{DRIVE}/data/camus_conversational.jsonl")) if p]

# conversational is small but shapes reply LENGTH (the anti-rambling set) -> upweight 3x
rows = sft + conv * 3
random.Random(3407).shuffle(rows)
print(f"sft={len(sft)}  conversational={len(conv)} (x3={len(conv)*3})  total={len(rows)}")
print("median response words:",
      sorted(len(r["response"].split()) for r in rows)[len(rows)//2])

In [ ]:
# 6) Format with the Gemma template (no system role — CORE lives in the RAG layer at runtime)
from datasets import Dataset

def fmt(r):
    return tokenizer.apply_chat_template(
        [{"role":"user","content":r["prompt"]},
         {"role":"assistant","content":r["response"]}],
        tokenize=False, add_generation_prompt=False)

ds = Dataset.from_list([{"text": fmt(r)} for r in rows])
print(ds[0]["text"][:400])
print("...\nrows:", len(ds))

In [ ]:
# ============================ CHECK 1 — file/þdata pre-flight ============================
assert len(sft) > 5000, f"sft corpus looks short: {len(sft)}"
assert len(conv) > 50, f"conversational corpus looks short: {len(conv)}"
bad = [r for r in rows if not r["prompt"].strip() or not r["response"].strip()]
assert not bad, f"{len(bad)} rows with empty prompt/response"
lens = [len(tokenizer(fmt(r))["input_ids"]) for r in rows[:400]]
print(f"sample token lengths: median={sorted(lens)[len(lens)//2]}, max={max(lens)}, "
      f"over MAX_SEQ={sum(l>MAX_SEQ for l in lens)}/400")
print("CHECK 1 PASSED")

In [ ]:
# 7) Trainer with response-only masking (Gemma markers)
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ,
        per_device_train_batch_size = 2,      # L4: keep 2; A100: can raise to 4
        gradient_accumulation_steps = 8,      # effective batch 16
        warmup_ratio = 0.03,
        num_train_epochs = 2,
        learning_rate = 1e-4,                 # voice pass on a fresh r=32 adapter
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/content/outputs",
        report_to = "none",
    ),
)

# Gemma-3 turn markers (NOT the Llama header ids)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part    = "<start_of_turn>model\n",
)
print("trainer ready")

In [ ]:
# ======================= CHECK 2 — response masking (THE critical gate) =======================
# Wrong offsets here silently train on the prompt. Verify on real batches.
import torch

def decode_pair(i):
    ex = trainer.train_dataset[i]
    ids = ex["input_ids"]; labels = ex["labels"]
    kept  = [t for t, l in zip(ids, labels) if l != -100]
    masked= [t for t, l in zip(ids, labels) if l == -100]
    return tokenizer.decode(masked), tokenizer.decode(kept)

ok = True
for i in [0, 1, 2, len(trainer.train_dataset)//2, len(trainer.train_dataset)-1]:
    masked_txt, kept_txt = decode_pair(i)
    # the trained span must be the MODEL turn only
    cond_a = "<start_of_turn>user" in masked_txt          # prompt is masked
    cond_b = "<start_of_turn>user" not in kept_txt        # prompt not trained
    cond_c = len(kept_txt.strip()) > 0                    # something IS trained
    print(f"[{i}] masked_ok={cond_a} kept_clean={cond_b} nonempty={cond_c}")
    print("    TRAINED:", repr(kept_txt[:120]))
    ok &= (cond_a and cond_b and cond_c)

assert ok, "MASKING BROKEN — do not train. Check instruction_part/response_part strings."
frac = float(sum((torch.tensor(trainer.train_dataset[i]['labels']) != -100).float().mean()
                 for i in range(50)) / 50)
print(f"\nmean trained-token fraction: {frac:.2f}  (expect ~0.2-0.6)")
assert 0.05 < frac < 0.9, "suspicious trained fraction"
print("CHECK 2 PASSED — training on model turns only")

In [ ]:
# 8) Train
stats = trainer.train()
print(stats)

In [ ]:
# ===================== CHECK 3 — behavioral smoke test (before saving) =====================
from transformers import TextStreamer
FastModel.for_inference(model)

def say(prompt, max_new_tokens=180):
    msgs = [{"role":"user","content":prompt}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7,
                         top_p=0.95, do_sample=True)
    txt = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return txt.strip()

tests = ["hey",
         "do you have a cat",
         "what do you think of the internet?",
         'Analyze this line: "The sea returned everything except the swimmer."']
for t in tests:
    print("="*70); print("USER:", t); print("CAMUS:", say(t))

print("\n" + "="*70)
print("""MANUAL GATE — read the four answers above:
  [ ] first person throughout (no 'Camus would...', no third person)
  [ ] no stage directions in parentheses (the gemma3 roleplay tic)
  [ ] not every answer ends with a question back (the gemma3 closer tic)
  [ ] greeting is SHORT; analysis engages instead of disowning
If two or more fail, stop and adjust data/epochs rather than proceeding.""")

In [ ]:
# 9) Save the voice adapter to Drive (INPUT to the Step-3.5 notebook)
model.save_pretrained(OUT_ADAPTER)
tokenizer.save_pretrained(OUT_ADAPTER)
import os
print("saved:", OUT_ADAPTER)
for f in sorted(os.listdir(OUT_ADAPTER)):
    print("  ", f, os.path.getsize(os.path.join(OUT_ADAPTER, f)))
assert os.path.exists(os.path.join(OUT_ADAPTER, "adapter_model.safetensors")), \
    "adapter weights missing!"
print("\nPhase 1 complete. Next: the Step-3.5 refusal-SFT notebook with\n"
      f"  SFT_ADAPTER = {OUT_ADAPTER}\n  BASE_MODEL  = {BASE_MODEL}")